# Ensemble vs Plain FastText — groceries vs shopping fix

Compares `rule → fastText` vs `rule → fastText → centroid → amount-rerank → LLM if <0.6` on the 548-row test set.
FastText alone: 95.1% valid, but groceries 0.745 prec. Ensemble should lift groceries via amount+centroid tie-break.

In [ ]:
import pathlib, csv, json
import sys
sys.path.insert(0, "..")
from normalize import normalize_merchant, merchant_for_training
from categorize_builtin import builtin_category
import fasttext

MODEL = pathlib.Path("../models/merchant_ft.bin")
model = fasttext.load_model(str(MODEL))
print("labels", model.labels)
# try centroid
try:
    from ensemble import centroid_scores, load_centroids
    load_centroids()
    print("centroid ready")
except Exception as e:
    print("centroid not available:", e)
    centroid_scores=lambda x: {}


In [ ]:
from ensemble import predict_ensemble
import pathlib
test_path = pathlib.Path("../data/processed/fasttext.test")
# fasttext.test is normalized; we need original merchant+amount for ensemble.
# Rebuild test set from labeled.csv split is complex, so we evaluate on a hand-crafted ambiguous set + full test via fastText directly.
ambiguous = [
    ("SAFEWAY #123     BURNABY", -3200, "groceries"),
    ("SAVE ON FOODS #2221     BURNABY", -4500, "groceries"),
    ("COSTCO WHOLESAL #123     BURNABY", -12000, "groceries"),  # large -> should stay groceries despite amount
    ("AMAZON.CA #123     TORONTO", -4500, "shopping"),
    ("AMAZON MARKETPLACE   TORONTO", -8999, "shopping"),
    ("WALMART SUPERCENTER  BURNABY", -4500, "groceries"),  # walmart can be groceries or shopping
    ("RANDOM GROCERY MART #123     VANCOUVER", -3200, "groceries"),
    ("UNKNOWN SHOP #999     TORONTO", -8000, "shopping"),
]
print(f"{'MERCHANT':<40} {'TRUTH':<12} {'PLAIN':<12} {'ENSEMBLE':<12} {'PROB':<5} SRC")
print("-"*110)
for mer, amt, truth in ambiguous:
    # plain
    txt=merchant_for_training(mer, amt)
    pl, pp = model.predict(txt, k=1)
    plain=pl[0].replace("__label__","") if pl else "?"
    plain_p=float(pp[0]) if len(pp) else 0
    # ensemble
    ens, ens_p, src = predict_ensemble(mer, amt, threshold=0.6, fasttext_model=model)
    print(f"{mer[:38]:<40} {truth:<12} {plain:<12} {str(ens):<12} {ens_p:.2f} {src}")


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import pathlib
# Full test set via fastText directly (from metrics already computed)
# Load metrics from evaluate.py
import json, pathlib
for name in ["test","valid","heldout_merchant"]:
    p=pathlib.Path(f"../data/processed/metrics_{name}.json")
    if p.exists():
        m=json.loads(p.read_text())
        print(f"{name}: acc={m['accuracy']:.3f} macro_f1={m['macro_f1']:.3f} low={m['low_conf_rate']:.1%} n={m['n']}")
        if name=="test":
            # show groceries row
            print(json.dumps(m['report']['groceries'], indent=2))


In [ ]:
# Ensemble full test (if you want to re-run without torch, this will be rule+fastText+amount only)
import csv, pathlib
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score
from ensemble import predict_ensemble
import csv, pathlib
# Build test set from labeled.csv + synthetic via same split logic as prepare.py but using ensemble
# Simpler: load labeled.csv and do a fresh 80/10/10 split in notebook and evaluate ensemble vs plain
import random, csv, pathlib
from collections import Counter
random.seed(42)
rows=list(csv.DictReader(open("../data/processed/labeled.csv", encoding="utf-8")))
# merge llm_augmented as prepare does
import pathlib
syn=pathlib.Path("../data/synthetic/llm_augmented.csv")
if syn.exists():
    rows+=list(csv.DictReader(syn.open(encoding="utf-8")))
print(f"total rows for notebook eval: {len(rows)}")
# simple random split 80/10/10
random.shuffle(rows)
n=len(rows)
test=rows[int(0.9*n):]
print(f"test {len(test)}", Counter(r['category_id'] for r in test))
y_true=[]; y_plain=[]; y_ens=[]
for r in test:
    mer=r["merchant_raw"]; cat=r["category_id"]
    try: amt=int(r["amount_cents"] or 0)
    except: amt=0
    y_true.append(cat)
    # plain
    txt=merchant_for_training(mer, amt)
    pl, pp = model.predict(txt, k=1)
    y_plain.append(pl[0].replace("__label__","") if pl else "__exclude")
    # ensemble
    ens, _, _ = predict_ensemble(mer, amt, threshold=0.6, fasttext_model=model)
    y_ens.append(ens or "__exclude")
from sklearn.metrics import accuracy_score, classification_report
print("\nPlain fastText")
print(classification_report(y_true, y_plain, zero_division=0, digits=3))
print("\nEnsemble")
print(classification_report(y_true, y_ens, zero_division=0, digits=3))
print(f"plain acc {accuracy_score(y_true, y_plain):.3f} vs ensemble {accuracy_score(y_true, y_ens):.3f}")


## Takeaway
- Plain fastText: groceries prec 0.745 (shopping bleed), avg_prob 0.78
- Ensemble adds `centroid` tie-break + `amount` bias (`large >$150 → shopping`, `medium $20-80 → groceries`) when `|p_groceries - p_shopping|<0.2`
- Expected: groceries prec → ~0.85, macro-F1 +0.01-0.02, latency still ~0ms (centroid 10ms) vs LLM 500ms
- If centroid not installed, ensemble degrades to `rule → fastText → amount-rerank` (still lifts groceries)